# Notebook 05 — Cohort & Retention Analysis

## Brazilian E-Commerce Public Dataset by Olist

### Objetivo

Analisar a evolução das **coortes de aquisição** do e-commerce Olist, medindo:

- retenção mensal;
- recompra acumulada;
- receita por safra;
- receita acumulada por cliente adquirido;
- diferenças associadas ao meio principal de pagamento da primeira compra.

Este notebook complementa o Notebook 04:

- o Notebook 04 responde **se o cliente recomprou dentro de uma janela fixa**;
- o Notebook 05 responde **como cada safra de aquisição evolui ao longo dos meses**.

### Perguntas de negócio

- Quantos clientes são adquiridos em cada mês?
- Qual percentual volta em M1, M2, M3, M6 e M12?
- A retenção melhorou ou piorou entre safras?
- Quanto da receita inicial de uma coorte reaparece nos meses seguintes?
- Qual o valor acumulado por cliente adquirido ao longo do tempo?
- O meio principal de pagamento da primeira compra está associado a padrões distintos de retenção?
- Quais coortes performam acima ou abaixo do benchmark?

> **Importante:** em e-commerce transacional, retenção mensal tende a ser baixa. Por isso, este notebook separa claramente **retenção mensal exata**, **recompra acumulada** e **valor acumulado**.


## 1. Configuração e carregamento

Este notebook depende explicitamente de:

- `orders_base.csv`, produzido no Notebook 01;
- `payment_behavior_order_level.csv`, produzido no Notebook 02.

Não há reconstrução alternativa das bases. Essa decisão mantém a arquitetura do projeto simples, rastreável e reproduzível.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

orders_base_path = PROCESSED_DIR / "orders_base.csv"
payment_behavior_path = (
    PROCESSED_DIR / "payment_behavior_order_level.csv"
)

if not orders_base_path.exists():
    raise FileNotFoundError(
        "orders_base.csv não encontrado. "
        "Execute primeiro o Notebook 01."
    )

if not payment_behavior_path.exists():
    raise FileNotFoundError(
        "payment_behavior_order_level.csv não encontrado. "
        "Execute primeiro o Notebook 02 revisado."
    )

orders_base = pd.read_csv(
    orders_base_path,
    parse_dates=["order_purchase_timestamp"]
)

payment_behavior = pd.read_csv(
    payment_behavior_path
)

print("orders_base:", orders_base.shape)
print("payment_behavior:", payment_behavior.shape)


## 2. Universo analítico e maturidade temporal

In [ ]:
required_cols = {
    "order_id",
    "customer_unique_id",
    "order_status",
    "order_purchase_timestamp",
    "payment_value",
}

missing = required_cols.difference(
    orders_base.columns
)

if missing:
    raise ValueError(
        f"Colunas obrigatórias ausentes em orders_base: {sorted(missing)}"
    )

delivered = (
    orders_base.loc[
        orders_base["order_status"].eq("delivered")
        & orders_base["customer_unique_id"].notna()
        & orders_base["order_purchase_timestamp"].notna()
    ]
    .copy()
)

delivered["payment_value"] = pd.to_numeric(
    delivered["payment_value"],
    errors="coerce",
)

delivered = (
    delivered
    .sort_values(
        [
            "customer_unique_id",
            "order_purchase_timestamp",
            "order_id",
        ]
    )
    .reset_index(drop=True)
)

delivered["order_month"] = (
    delivered["order_purchase_timestamp"]
    .dt.to_period("M")
)

dataset_start = (
    delivered["order_purchase_timestamp"].min()
)

dataset_end = (
    delivered["order_purchase_timestamp"].max()
)

last_observed_month = (
    dataset_end.to_period("M")
)

month_end = (
    dataset_end
    + pd.offsets.MonthEnd(0)
)

if dataset_end.normalize() < month_end.normalize():
    last_complete_month = (
        last_observed_month - 1
    )
else:
    last_complete_month = (
        last_observed_month
    )

print("Período entregue:", dataset_start, "→", dataset_end)
print("Último mês observado:", last_observed_month)
print("Último mês completo:", last_complete_month)
print("Pedidos entregues:", f"{len(delivered):,}")
print(
    "Clientes únicos:",
    f"{delivered['customer_unique_id'].nunique():,}"
)


### Governança financeira

A retenção de clientes usa todos os pedidos entregues válidos.

As análises financeiras utilizam somente pedidos com `payment_value`:

- não nulo;
- maior ou igual a zero.

Ausência de `payment_value` não é interpretada como receita zero.


In [ ]:
payment_quality = pd.Series({
    "delivered_orders":
        len(delivered),
    "missing_payment_value":
        int(
            delivered[
                "payment_value"
            ].isna().sum()
        ),
    "negative_payment_value":
        int(
            delivered[
                "payment_value"
            ].lt(0).sum()
        ),
})

payment_quality


In [ ]:
delivered_financial = (
    delivered.loc[
        delivered["payment_value"].notna()
        & delivered["payment_value"].ge(0)
    ]
    .copy()
)

print(
    "Pedidos válidos para análises financeiras:",
    f"{len(delivered_financial):,}"
)


## 3. Definição das coortes

A coorte de cada cliente é o **mês da primeira compra entregue**.

O `cohort_index` mede quantos meses se passaram desde a aquisição:

- `M0`: mês da primeira compra;
- `M1`: mês seguinte;
- `M2`: dois meses depois;
- etc.


In [ ]:
first_purchase_month = (
    delivered
    .groupby(
        "customer_unique_id"
    )["order_month"]
    .min()
    .rename("cohort_month")
)

delivered = (
    delivered
    .merge(
        first_purchase_month,
        on="customer_unique_id",
        how="left",
        validate="many_to_one",
    )
)

delivered_financial = (
    delivered_financial
    .merge(
        first_purchase_month,
        on="customer_unique_id",
        how="left",
        validate="many_to_one",
    )
)

def month_diff(month_a, month_b):
    return (
        (month_a.dt.year - month_b.dt.year) * 12
        + (month_a.dt.month - month_b.dt.month)
    )

delivered["cohort_index"] = (
    month_diff(
        delivered["order_month"],
        delivered["cohort_month"],
    )
    .astype(int)
)

delivered_financial["cohort_index"] = (
    month_diff(
        delivered_financial["order_month"],
        delivered_financial["cohort_month"],
    )
    .astype(int)
)

assert delivered["cohort_index"].ge(0).all()

display(
    delivered[
        [
            "customer_unique_id",
            "order_id",
            "order_month",
            "cohort_month",
            "cohort_index",
        ]
    ].head()
)


## 4. Aquisição por coorte

In [ ]:
cohort_sizes = (
    first_purchase_month
    .reset_index()
    .groupby(
        "cohort_month"
    )
    .agg(
        acquired_customers=(
            "customer_unique_id",
            "nunique",
        )
    )
    .reset_index()
    .sort_values(
        "cohort_month"
    )
)

cohort_sizes["cohort_month_str"] = (
    cohort_sizes["cohort_month"].astype(str)
)

display(cohort_sizes)

fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(
    cohort_sizes["cohort_month_str"],
    cohort_sizes["acquired_customers"],
)

ax.set_title(
    "Clientes adquiridos por coorte mensal"
)
ax.set_xlabel(
    "Mês da primeira compra"
)
ax.set_ylabel(
    "Clientes adquiridos"
)

plt.xticks(rotation=60)
plt.tight_layout()
plt.show()


## 5. Matriz de retenção mensal de clientes

A retenção em `Mk` é a proporção de clientes da coorte que fizeram **pelo menos uma compra naquele mês específico**.

Para evitar censura à direita:

- meses futuros são exibidos como `NaN`;
- a maturidade é calculada usando o **último mês calendário completo**, não o último mês parcialmente observado.


In [ ]:
cohort_customers = (
    delivered
    .groupby(
        [
            "cohort_month",
            "cohort_index",
        ]
    )["customer_unique_id"]
    .nunique()
    .unstack(fill_value=0)
    .sort_index()
)

max_age = int(
    max(
        (
            last_complete_month.year - cm.year
        ) * 12
        + (
            last_complete_month.month - cm.month
        )
        for cm in cohort_customers.index
    )
)

cohort_customers = (
    cohort_customers
    .reindex(
        columns=range(max_age + 1),
        fill_value=0,
    )
)

cohort_size_series = (
    cohort_customers[0].copy()
)

customer_retention = (
    cohort_customers
    .div(
        cohort_size_series,
        axis=0,
    )
)

for cm in customer_retention.index:
    observed_age = (
        (last_complete_month.year - cm.year) * 12
        + (last_complete_month.month - cm.month)
    )

    customer_retention.loc[
        cm,
        customer_retention.columns > observed_age
    ] = np.nan

customer_retention.index = (
    customer_retention.index.astype(str)
)

customer_retention.index.name = (
    "cohort_month"
)

customer_retention.columns = [
    f"M{i}"
    for i in customer_retention.columns
]

display(
    (
        customer_retention.iloc[:, :13]
        * 100
    ).round(2)
)


In [ ]:
retention_plot = (
    customer_retention.iloc[:, :13]
    * 100
)

fig, ax = plt.subplots(figsize=(14, 8))

img = ax.imshow(
    retention_plot.values,
    aspect="auto",
)

fig.colorbar(
    img,
    ax=ax,
    label="Retenção (%)",
)

ax.set_title(
    "Heatmap de retenção mensal por coorte — M0 a M12"
)
ax.set_xlabel(
    "Idade da coorte"
)
ax.set_ylabel(
    "Coorte de aquisição"
)

ax.set_xticks(
    range(
        len(retention_plot.columns)
    ),
    retention_plot.columns,
)

ax.set_yticks(
    range(
        len(retention_plot.index)
    ),
    retention_plot.index,
)

plt.tight_layout()
plt.show()


## 6. Curva agregada de retenção — clientes elegíveis

Para cada idade da coorte, calculamos a retenção considerando apenas clientes de coortes maduras o suficiente para que aquele mês estivesse completamente observável.


In [ ]:
customer_cohort = (
    first_purchase_month
    .reset_index()
)

customer_cohort[
    "cohort_age_complete_months"
] = (
    (
        last_complete_month.year
        - customer_cohort[
            "cohort_month"
        ].dt.year
    ) * 12
    +
    (
        last_complete_month.month
        - customer_cohort[
            "cohort_month"
        ].dt.month
    )
)

activity = (
    delivered[
        [
            "customer_unique_id",
            "cohort_index",
        ]
    ]
    .drop_duplicates()
)

weighted_retention_rows = []

for month_index in range(
    0,
    min(max_age, 18) + 1
):
    eligible_ids = (
        customer_cohort.loc[
            customer_cohort[
                "cohort_age_complete_months"
            ].ge(month_index),
            "customer_unique_id",
        ]
    )

    active_ids = set(
        activity.loc[
            activity[
                "cohort_index"
            ].eq(month_index),
            "customer_unique_id",
        ]
    )

    retained = (
        eligible_ids
        .isin(active_ids)
        .sum()
    )

    weighted_retention_rows.append({
        "cohort_index":
            month_index,
        "eligible_customers":
            len(eligible_ids),
        "active_customers":
            int(retained),
        "retention_rate":
            retained / len(eligible_ids)
            if len(eligible_ids)
            else np.nan,
    })

weighted_retention = pd.DataFrame(
    weighted_retention_rows
)

display(
    weighted_retention.head(13)
)


In [ ]:
plot_wr = (
    weighted_retention.loc[
        weighted_retention[
            "cohort_index"
        ].le(12)
    ]
)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    plot_wr["cohort_index"],
    plot_wr["retention_rate"] * 100,
    marker="o",
)

ax.set_title(
    "Retenção mensal agregada — clientes elegíveis"
)
ax.set_xlabel(
    "Meses desde a aquisição"
)
ax.set_ylabel(
    "Retenção (%)"
)

ax.set_xticks(
    plot_wr["cohort_index"]
)

plt.tight_layout()
plt.show()


### Retenção mensal não é recompra acumulada

Um cliente pode:

- comprar em M1;
- não comprar em M2;
- voltar em M3.

Por isso:

- **retenção mensal** responde “comprou exatamente em Mk?”;
- **recompra acumulada** responde “já realizou uma segunda compra até Mk?”.


## 7. Recompra acumulada por idade da coorte

Para manter o mesmo denominador ao longo da curva acumulada, usamos apenas clientes com **12 meses completos de observação**.

Assim, M1, M3, M6 e M12 são comparados sobre a mesma população madura.


In [ ]:
ordered = (
    delivered
    .sort_values(
        [
            "customer_unique_id",
            "order_purchase_timestamp",
            "order_id",
        ]
    )
    .copy()
)

ordered["order_number"] = (
    ordered
    .groupby(
        "customer_unique_id"
    )
    .cumcount()
    + 1
)

second_purchase = (
    ordered.loc[
        ordered[
            "order_number"
        ].eq(2),
        [
            "customer_unique_id",
            "order_month",
            "cohort_month",
        ],
    ]
    .copy()
)

second_purchase[
    "second_purchase_index"
] = (
    (
        second_purchase[
            "order_month"
        ].dt.year
        - second_purchase[
            "cohort_month"
        ].dt.year
    ) * 12
    +
    (
        second_purchase[
            "order_month"
        ].dt.month
        - second_purchase[
            "cohort_month"
        ].dt.month
    )
).astype(int)

customer_repeat_age = (
    customer_cohort
    .merge(
        second_purchase[
            [
                "customer_unique_id",
                "second_purchase_index",
            ]
        ],
        on="customer_unique_id",
        how="left",
        validate="one_to_one",
    )
)

FOLLOWUP_MONTHS = 12

mature_repeat_base = (
    customer_repeat_age.loc[
        customer_repeat_age[
            "cohort_age_complete_months"
        ].ge(FOLLOWUP_MONTHS)
    ]
    .copy()
)

cumulative_repeat_rows = []

for month_index in range(
    0,
    FOLLOWUP_MONTHS + 1
):
    repeated = (
        mature_repeat_base[
            "second_purchase_index"
        ].notna()
        & mature_repeat_base[
            "second_purchase_index"
        ].le(month_index)
    )

    cumulative_repeat_rows.append({
        "cohort_index":
            month_index,
        "eligible_customers":
            len(mature_repeat_base),
        "customers_with_second_purchase":
            int(repeated.sum()),
        "cumulative_repeat_rate":
            repeated.mean(),
    })

cumulative_repeat = pd.DataFrame(
    cumulative_repeat_rows
)

display(cumulative_repeat)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    cumulative_repeat["cohort_index"],
    cumulative_repeat[
        "cumulative_repeat_rate"
    ] * 100,
    marker="o",
)

ax.set_title(
    "Recompra acumulada — clientes com 12 meses completos de observação"
)
ax.set_xlabel(
    "Meses desde a aquisição"
)
ax.set_ylabel(
    "Clientes com segunda compra acumulada (%)"
)

ax.set_xticks(
    cumulative_repeat[
        "cohort_index"
    ]
)

plt.tight_layout()
plt.show()


In [ ]:
mature_population_kpis = pd.Series({
    "followup_months":
        FOLLOWUP_MONTHS,
    "mature_customers":
        len(mature_repeat_base),
    "mature_cohorts":
        mature_repeat_base[
            "cohort_month"
        ].nunique(),
})

mature_population_kpis


## 8. Comparação de M1, M2, M3, M6 e M12 por safra

In [ ]:
ret_long = (
    customer_retention
    .reset_index()
    .melt(
        id_vars="cohort_month",
        var_name="month_label",
        value_name="retention_rate",
    )
)

ret_long["cohort_index"] = (
    ret_long[
        "month_label"
    ]
    .str.replace(
        "M",
        "",
        regex=False,
    )
    .astype(int)
)

cohort_comparison = (
    cohort_sizes[
        [
            "cohort_month",
            "acquired_customers",
        ]
    ]
    .copy()
)

cohort_comparison[
    "cohort_month"
] = (
    cohort_comparison[
        "cohort_month"
    ].astype(str)
)

for k in [1, 2, 3, 6, 12]:
    temp = (
        ret_long.loc[
            ret_long[
                "cohort_index"
            ].eq(k),
            [
                "cohort_month",
                "retention_rate",
            ],
        ]
        .rename(
            columns={
                "retention_rate":
                    f"retention_M{k}"
            }
        )
    )

    cohort_comparison = (
        cohort_comparison
        .merge(
            temp,
            on="cohort_month",
            how="left",
        )
    )

display(cohort_comparison)


## 9. Benchmark de retenção por coorte

In [ ]:
def weighted_retention_at(k):
    row = (
        weighted_retention.loc[
            weighted_retention[
                "cohort_index"
            ].eq(k)
        ]
    )

    return (
        row["retention_rate"].iloc[0]
        if len(row)
        else np.nan
    )

for k in [1, 3, 6]:
    benchmark = weighted_retention_at(k)

    cohort_comparison[
        f"delta_M{k}_vs_benchmark"
    ] = (
        cohort_comparison[
            f"retention_M{k}"
        ]
        - benchmark
    )

display(cohort_comparison)


### Uso executivo

O benchmark ajuda a responder:

> “Esta coorte performou melhor ou pior do que a média das coortes elegíveis na mesma idade?”

Isso é mais informativo do que comparar taxas isoladas.


## 10. Revenue Index por coorte

O **Revenue Index** compara a receita mensal da coorte com a receita gerada no mês de aquisição:

\[
\text{Revenue Index}_{M_k}
=
\frac{
\text{Receita da coorte em } M_k
}{
\text{Receita da coorte em } M_0
}
\]

O indicador:

- não mede percentual de clientes retidos;
- não é SaaS NRR;
- pode superar 100% se a coorte gerar mais receita em um mês posterior do que em M0.


In [ ]:
cohort_revenue = (
    delivered_financial
    .groupby(
        [
            "cohort_month",
            "cohort_index",
        ]
    )["payment_value"]
    .sum()
    .unstack(fill_value=0)
    .sort_index()
    .reindex(
        columns=range(max_age + 1),
        fill_value=0,
    )
)

m0_revenue = cohort_revenue[0].replace(
    0,
    np.nan,
)

revenue_retention = (
    cohort_revenue
    .div(
        m0_revenue,
        axis=0,
    )
)

for cm in revenue_retention.index:
    observed_age = (
        (last_complete_month.year - cm.year) * 12
        + (last_complete_month.month - cm.month)
    )

    revenue_retention.loc[
        cm,
        revenue_retention.columns > observed_age
    ] = np.nan

revenue_retention.index = (
    revenue_retention.index.astype(str)
)

revenue_retention.index.name = (
    "cohort_month"
)

revenue_retention.columns = [
    f"M{i}"
    for i in revenue_retention.columns
]

display(
    (
        revenue_retention.iloc[:, :13]
        * 100
    ).round(2)
)


In [ ]:
revenue_plot = (
    revenue_retention.iloc[:, :13]
    * 100
)

fig, ax = plt.subplots(figsize=(14, 8))

img = ax.imshow(
    revenue_plot.values,
    aspect="auto",
)

fig.colorbar(
    img,
    ax=ax,
    label="Revenue Index (%)",
)

ax.set_title(
    "Heatmap do Revenue Index por coorte — M0 a M12"
)
ax.set_xlabel(
    "Idade da coorte"
)
ax.set_ylabel(
    "Coorte de aquisição"
)

ax.set_xticks(
    range(
        len(revenue_plot.columns)
    ),
    revenue_plot.columns,
)

ax.set_yticks(
    range(
        len(revenue_plot.index)
    ),
    revenue_plot.index,
)

plt.tight_layout()
plt.show()


## 11. Receita acumulada por cliente adquirido — proxy de LTV observado

A receita acumulada por cliente adquirido é uma **proxy descritiva de LTV observado**.

Ela não representa CLV predito porque o dataset não contém:

- margem;
- CAC;
- horizonte futuro;
- taxa de desconto;
- custo promocional.


In [ ]:
cohort_revenue_raw = (
    delivered_financial
    .groupby(
        [
            "cohort_month",
            "cohort_index",
        ]
    )["payment_value"]
    .sum()
    .unstack(fill_value=0)
    .sort_index()
    .reindex(
        columns=range(max_age + 1),
        fill_value=0,
    )
)

cohort_sizes_period = (
    first_purchase_month
    .value_counts()
    .sort_index()
)

cumulative_revenue_per_customer = (
    cohort_revenue_raw
    .cumsum(axis=1)
    .div(
        cohort_sizes_period,
        axis=0,
    )
)

for cm in cumulative_revenue_per_customer.index:
    observed_age = (
        (last_complete_month.year - cm.year) * 12
        + (last_complete_month.month - cm.month)
    )

    cumulative_revenue_per_customer.loc[
        cm,
        cumulative_revenue_per_customer.columns > observed_age
    ] = np.nan

cumulative_revenue_per_customer.index = (
    cumulative_revenue_per_customer.index.astype(str)
)

cumulative_revenue_per_customer.index.name = (
    "cohort_month"
)

cumulative_revenue_per_customer.columns = [
    f"M{i}"
    for i in cumulative_revenue_per_customer.columns
]

display(
    cumulative_revenue_per_customer.iloc[:, :13]
    .round(2)
)


In [ ]:
MIN_COHORT_SIZE = 300

eligible_curves = (
    cohort_comparison.loc[
        cohort_comparison[
            "acquired_customers"
        ].ge(MIN_COHORT_SIZE)
        & cohort_comparison[
            "retention_M6"
        ].notna(),
        "cohort_month",
    ]
    .tolist()
)

if len(eligible_curves) > 5:
    idx = (
        np.linspace(
            0,
            len(eligible_curves) - 1,
            5,
        )
        .round()
        .astype(int)
    )

    selected_cohorts = [
        eligible_curves[i]
        for i in idx
    ]
else:
    selected_cohorts = eligible_curves

fig, ax = plt.subplots(figsize=(10, 6))

for cohort in selected_cohorts:
    row = (
        cumulative_revenue_per_customer.loc[
            cohort,
            [
                f"M{i}"
                for i in range(0, 7)
            ],
        ]
    )

    ax.plot(
        range(0, 7),
        row.values,
        marker="o",
        label=cohort,
    )

ax.set_title(
    "Receita acumulada por cliente adquirido — coortes selecionadas"
)
ax.set_xlabel(
    "Meses desde a aquisição"
)
ax.set_ylabel(
    "Receita acumulada por cliente (R$)"
)

ax.set_xticks(
    range(0, 7)
)

if selected_cohorts:
    ax.legend(
        title="Coorte"
    )

plt.tight_layout()
plt.show()


## 12. Meio principal da primeira compra × retenção

In [ ]:
required_payment_cols = {
    "order_id",
    "primary_payment_type",
}

missing_payment_cols = (
    required_payment_cols
    .difference(
        payment_behavior.columns
    )
)

if missing_payment_cols:
    raise ValueError(
        "Colunas ausentes em payment_behavior_order_level.csv: "
        f"{sorted(missing_payment_cols)}"
    )

first_orders = (
    ordered.loc[
        ordered[
            "order_number"
        ].eq(1),
        [
            "customer_unique_id",
            "order_id",
            "cohort_month",
        ],
    ]
    .rename(
        columns={
            "order_id": "first_order_id"
        }
    )
)

customer_acquisition = (
    customer_cohort
    .merge(
        first_orders[
            [
                "customer_unique_id",
                "first_order_id",
            ]
        ],
        on="customer_unique_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        payment_behavior[
            [
                "order_id",
                "primary_payment_type",
            ]
        ].rename(
            columns={
                "order_id":
                    "first_order_id"
            }
        ),
        on="first_order_id",
        how="left",
        validate="one_to_one",
    )
)

customer_acquisition[
    "primary_payment_type"
] = (
    customer_acquisition[
        "primary_payment_type"
    ]
    .fillna("unknown")
)

display(
    customer_acquisition[
        "primary_payment_type"
    ]
    .value_counts(
        dropna=False
    )
    .to_frame(
        "customers"
    )
)


In [ ]:
retention_payment_rows = []

for payment_type, group in (
    customer_acquisition
    .groupby(
        "primary_payment_type"
    )
):
    for month_index in range(1, 7):
        eligible = (
            group.loc[
                group[
                    "cohort_age_complete_months"
                ].ge(month_index),
                "customer_unique_id",
            ]
        )

        active_ids = set(
            activity.loc[
                activity[
                    "cohort_index"
                ].eq(month_index),
                "customer_unique_id",
            ]
        )

        retained = (
            eligible
            .isin(active_ids)
            .sum()
        )

        retention_payment_rows.append({
            "primary_payment_type":
                payment_type,
            "cohort_index":
                month_index,
            "eligible_customers":
                len(eligible),
            "active_customers":
                int(retained),
            "retention_rate":
                retained / len(eligible)
                if len(eligible)
                else np.nan,
        })

retention_by_payment = pd.DataFrame(
    retention_payment_rows
)

display(retention_by_payment)


In [ ]:
MIN_PAYMENT_CUSTOMERS = 500

payment_sizes = (
    customer_acquisition[
        "primary_payment_type"
    ]
    .value_counts()
)

major_payment_types = (
    payment_sizes.loc[
        payment_sizes.ge(
            MIN_PAYMENT_CUSTOMERS
        )
    ]
    .index
    .tolist()
)

plot_pay = (
    retention_by_payment.loc[
        retention_by_payment[
            "primary_payment_type"
        ].isin(
            major_payment_types
        )
        & retention_by_payment[
            "primary_payment_type"
        ].ne("unknown")
    ]
)

fig, ax = plt.subplots(figsize=(10, 6))

for payment_type, group in (
    plot_pay
    .groupby(
        "primary_payment_type"
    )
):
    ax.plot(
        group["cohort_index"],
        group["retention_rate"] * 100,
        marker="o",
        label=payment_type,
    )

ax.set_title(
    "Retenção mensal por meio principal da primeira compra"
)
ax.set_xlabel(
    "Meses desde a aquisição"
)
ax.set_ylabel(
    "Retenção (%)"
)

ax.set_xticks(
    range(1, 7)
)

if not plot_pay.empty:
    ax.legend(
        title="Pagamento principal"
    )

plt.tight_layout()
plt.show()


## 13. Resumo executivo por coorte

In [ ]:
acquisition_revenue = (
    delivered_financial.loc[
        delivered_financial[
            "cohort_index"
        ].eq(0)
    ]
    .groupby(
        "cohort_month"
    )["payment_value"]
    .sum()
    .rename(
        "acquisition_month_revenue"
    )
    .reset_index()
)

cohort_summary = (
    cohort_sizes[
        [
            "cohort_month",
            "acquired_customers",
        ]
    ]
    .copy()
    .merge(
        acquisition_revenue,
        on="cohort_month",
        how="left",
    )
)

cohort_summary[
    "avg_acquisition_month_revenue_per_customer"
] = (
    cohort_summary[
        "acquisition_month_revenue"
    ]
    / cohort_summary[
        "acquired_customers"
    ]
)

cohort_summary[
    "cohort_month"
] = (
    cohort_summary[
        "cohort_month"
    ].astype(str)
)

for k in [1, 2, 3, 6, 12]:
    temp = (
        ret_long.loc[
            ret_long[
                "cohort_index"
            ].eq(k),
            [
                "cohort_month",
                "retention_rate",
            ],
        ]
        .rename(
            columns={
                "retention_rate":
                    f"retention_M{k}"
            }
        )
    )

    cohort_summary = (
        cohort_summary
        .merge(
            temp,
            on="cohort_month",
            how="left",
        )
    )

for k in [1, 3, 6]:
    benchmark = weighted_retention_at(k)

    cohort_summary[
        f"delta_M{k}_vs_benchmark"
    ] = (
        cohort_summary[
            f"retention_M{k}"
        ]
        - benchmark
    )

for k in [3, 6, 12]:
    col = f"M{k}"

    if col in cumulative_revenue_per_customer.columns:
        temp = (
            cumulative_revenue_per_customer[
                [col]
            ]
            .reset_index()
            .rename(
                columns={
                    col:
                        f"cum_revenue_per_customer_M{k}"
                }
            )
        )

        cohort_summary = (
            cohort_summary
            .merge(
                temp,
                on="cohort_month",
                how="left",
            )
        )

display(cohort_summary)


## 14. KPIs executivos

In [ ]:
def cumulative_repeat_at(k):
    row = (
        cumulative_repeat.loc[
            cumulative_repeat[
                "cohort_index"
            ].eq(k)
        ]
    )

    return (
        row[
            "cumulative_repeat_rate"
        ].iloc[0]
        if len(row)
        else np.nan
    )

def eligible_customers_at(k):
    row = (
        weighted_retention.loc[
            weighted_retention[
                "cohort_index"
            ].eq(k)
        ]
    )

    return (
        int(
            row[
                "eligible_customers"
            ].iloc[0]
        )
        if len(row)
        else 0
    )

repeat_customers = int(
    second_purchase[
        "customer_unique_id"
    ].nunique()
)

raw_repeat_rate = (
    repeat_customers
    / delivered[
        "customer_unique_id"
    ].nunique()
)

executive_kpis = pd.DataFrame({
    "kpi": [
        "customers_delivered",
        "cohorts_observed",
        "raw_repeat_rate_any_time",
        "retention_M1_weighted",
        "eligible_customers_M1",
        "retention_M3_weighted",
        "eligible_customers_M3",
        "retention_M6_weighted",
        "eligible_customers_M6",
        "retention_M12_weighted",
        "eligible_customers_M12",
        "cumulative_repeat_by_M1",
        "cumulative_repeat_by_M3",
        "cumulative_repeat_by_M6",
        "cumulative_repeat_by_M12",
        "mature_customers_12m",
    ],
    "value": [
        delivered[
            "customer_unique_id"
        ].nunique(),
        cohort_sizes[
            "cohort_month"
        ].nunique(),
        raw_repeat_rate,
        weighted_retention_at(1),
        eligible_customers_at(1),
        weighted_retention_at(3),
        eligible_customers_at(3),
        weighted_retention_at(6),
        eligible_customers_at(6),
        weighted_retention_at(12),
        eligible_customers_at(12),
        cumulative_repeat_at(1),
        cumulative_repeat_at(3),
        cumulative_repeat_at(6),
        cumulative_repeat_at(12),
        len(mature_repeat_base),
    ],
})

display(executive_kpis)


## 15. Metadata temporal

In [ ]:
cohort_metadata = pd.DataFrame({
    "metric": [
        "dataset_start",
        "dataset_end",
        "last_observed_month",
        "last_complete_month",
        "followup_months_cumulative_repeat",
    ],
    "value": [
        str(dataset_start),
        str(dataset_end),
        str(last_observed_month),
        str(last_complete_month),
        str(FOLLOWUP_MONTHS),
    ],
})

cohort_metadata


## 16. Leitura executiva

In [ ]:
print("LEITURA EXECUTIVA")
print("-" * 90)

print(
    f"1. A base entregue reúne "
    f"{delivered['customer_unique_id'].nunique():,} clientes "
    f"em {cohort_sizes['cohort_month'].nunique()} coortes mensais."
)

print(
    f"2. A retenção ponderada é "
    f"{weighted_retention_at(1):.2%} em M1, "
    f"{weighted_retention_at(3):.2%} em M3 e "
    f"{weighted_retention_at(6):.2%} em M6."
)

print(
    f"3. Entre clientes com 12 meses completos de observação, "
    f"a recompra acumulada chega a "
    f"{cumulative_repeat_at(1):.2%} até M1, "
    f"{cumulative_repeat_at(3):.2%} até M3 e "
    f"{cumulative_repeat_at(6):.2%} até M6."
)

print(
    f"4. A taxa observada de clientes com pelo menos dois "
    f"pedidos entregues é {raw_repeat_rate:.2%}."
)

print(
    "5. Retenção mensal e recompra acumulada respondem "
    "perguntas diferentes e devem aparecer separadamente."
)

print(
    f"6. O último mês completo utilizado para maturidade "
    f"das coortes é {last_complete_month}."
)

print(
    "7. Receita acumulada por cliente ajuda a avaliar "
    "a qualidade econômica das safras sem confundir "
    "frequência mensal com valor gerado."
)


## 17. Recomendações para storytelling executivo

### Tese 1 — Aquisição e relacionamento devem ser lidos separadamente

O volume de novos clientes não deve ser interpretado como melhora automática da retenção.

### Tese 2 — A maturidade das coortes precisa ser respeitada

Coortes recentes só devem ser comparadas em idades já completamente observáveis.

### Tese 3 — Retenção mensal e recompra acumulada são complementares

A primeira mostra atividade em um mês específico; a segunda mostra conversão acumulada à segunda compra.

### Tese 4 — Valor econômico deve acompanhar retenção

O Revenue Index e a receita acumulada por cliente adquirido ajudam a distinguir coortes com baixa frequência mensal, mas valor relevante.

### Tese 5 — Pagamento é sinal de segmentação, não causa

Diferenças de retenção por meio principal podem refletir categoria, ticket e perfil do cliente.

### Ação sugerida

Acompanhar cada nova safra por:

- M1;
- M3;
- M6;
- recompra acumulada;
- receita acumulada por cliente;
- delta versus benchmark.

Esse scorecard é mais informativo do que olhar apenas pedidos ou GMV total.


## 18. Sanity checks

In [ ]:
assert delivered[
    "cohort_index"
].ge(0).all(), (
    "Há cohort_index negativo."
)

assert (
    last_complete_month
    <= last_observed_month
)

assert np.allclose(
    customer_retention[
        "M0"
    ].dropna(),
    1.0,
), "M0 da retenção deve ser 100%."

if customer_retention.shape[1] > 1:
    assert (
        customer_retention
        .drop(
            columns=["M0"]
        )
        .stack()
        .between(0, 1)
        .all()
    ), "Retenção fora de [0,1]."

assert (
    weighted_retention[
        "retention_rate"
    ]
    .dropna()
    .between(0, 1)
    .all()
)

assert (
    cumulative_repeat[
        "cumulative_repeat_rate"
    ]
    .between(0, 1)
    .all()
)

assert (
    cumulative_repeat[
        "cumulative_repeat_rate"
    ]
    .is_monotonic_increasing
), (
    "Com denominador fixo, a recompra acumulada "
    "deve ser não decrescente."
)

assert (
    second_purchase[
        "second_purchase_index"
    ]
    .ge(0)
    .all()
)

assert customer_acquisition[
    "customer_unique_id"
].is_unique

assert cohort_summary[
    "cohort_month"
].is_unique

assert (
    cohort_sizes[
        "acquired_customers"
    ].sum()
    == delivered[
        "customer_unique_id"
    ].nunique()
)

assert delivered_financial[
    "payment_value"
].ge(0).all()

numeric_cum = (
    cumulative_revenue_per_customer
    .select_dtypes(
        include="number"
    )
    .to_numpy()
)

assert not np.isinf(
    numeric_cum
).any()

print(
    "Sanity checks concluídos com sucesso."
)


## 19. Exportação das camadas analíticas

In [ ]:
customer_retention_export = (
    customer_retention
    .reset_index()
)

revenue_retention_export = (
    revenue_retention
    .reset_index()
)

cumulative_revenue_export = (
    cumulative_revenue_per_customer
    .reset_index()
)

exports = {
    "cohort_customer_retention_matrix.csv":
        customer_retention_export,
    "cohort_weighted_retention_curve.csv":
        weighted_retention,
    "cohort_cumulative_repeat_curve.csv":
        cumulative_repeat,
    "cohort_revenue_index_matrix.csv":
        revenue_retention_export,
    "cohort_cumulative_revenue_per_customer.csv":
        cumulative_revenue_export,
    "cohort_retention_by_first_payment.csv":
        retention_by_payment,
    "cohort_summary.csv":
        cohort_summary,
    "cohort_executive_kpis.csv":
        executive_kpis,
    "cohort_metadata.csv":
        cohort_metadata,
    "cohort_benchmark_comparison.csv":
        cohort_comparison,
}

for filename, df in exports.items():
    path = (
        PROCESSED_DIR
        / filename
    )

    df.to_csv(
        path,
        index=False,
    )

    print(
        "Exportado:",
        path,
        df.shape,
    )


## Conclusão

O Notebook 05 transforma a base de pedidos em uma visão longitudinal de aquisição, retenção e valor por safra.

As principais saídas são:

1. **Heatmap de retenção mensal** com maturidade baseada no último mês completo;
2. **Curva de recompra acumulada** com denominador fixo para coortes maduras;
3. **Revenue Index** por coorte;
4. **Receita acumulada por cliente adquirido** como proxy descritiva de LTV observado;
5. **Retenção por meio principal de pagamento**;
6. **Scorecard por coorte** com benchmark e deltas;
7. **KPIs com denominadores explícitos** para uso em Power BI.

O próximo passo é o **Notebook 06 — Repurchase Prediction**, usando a base preparada no Notebook 04 para estimar a probabilidade de recompra em 90 dias com validação temporal.
